**1. Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


**2. Load the 5G NIDD Dataset**

In [ ]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',        # drop these columns as they dont have any importance in model(these are IP addresses)
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])   # keeps only numeric features

print("After numeric selection:", X.shape)


Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


**3. Data Cleaning and Handling Missing Values**

In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)     # remove infinite values
X.fillna(0, inplace=True)                              # handle missing values


**4. Feature Selection using SelectKBest**

In [ ]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


**5. Label Encoding(Encode Target Labels)**

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)


Classes: 20


**6. Split Dataset into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,                    # train data = 80%, test data = 20%
    stratify=y_encoded,
    random_state=42
)


**7. Feature Scaling using StandardScaler**

In [ ]:
# scaler = StandardScaler()

# X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
# X_test  = scaler.transform(X_test)        # TRANSFORM TEST


In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(1000, len(X_train)),   # avoids warning if dataset < 1000
    output_distribution='normal',          # 'normal' usually works better for neural networks
    random_state=42
)

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST

**8. Reshape Data for BiGRU Input**

In [ ]:
X_train = X_train.reshape(-1, 36, 1)        # Prepare data in 3D format required by GRU networks
X_test  = X_test.reshape(-1, 36, 1)


**9. Import Deep Learning Layers and Mobile-Net-V1 & BIGRU(Combined) MODEL**

In [ ]:
def MobileNetV1_BiGRU(drop_rate=0.5, gru_units=128, dense_units=256):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----                    # MOBILE-NET V1 MODEL
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----                                 # BIGRU MODEL
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----                                    # PROJECTION(Combines both models)
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


**10. Install keras tuner**

In [ ]:
!pip install keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.5 MB/s eta 0:00:00


**11. Setup TensorFlow & Focal Loss**

In [ ]:
import tensorflow as tf

def focal_loss(gamma=2.0, alpha=0.25):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


**12. Handle Class Imbalance using Class Weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


**13. Hyperparameter Tuning for Combined Model**

In [ ]:
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping

def build_model(hp):

    model = MobileNetV1_BiGRU(
        drop_rate = hp.Choice("dropout",[0.3,0.5,0.6]),            # fixed 3 values for each feature of hyperparameter tuning
        gru_units = hp.Choice("gru_units",[64,128,256]),           # 4 hyperparameters - GRU units, dropout, dense units, LR
        dense_units = hp.Choice("dense",[128,256,512])
    )

    lr = hp.Choice("lr",[1e-2,1e-3,1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),       # Adam optimizer
        loss=focal_loss(gamma=2, alpha=0.25),
        metrics=["accuracy"]
    )

    return model


tuner = kt.Hyperband(             # tuning using Hyperband tuner
    build_model,
    objective="val_accuracy",
    max_epochs=7,   # normally 10      # runs for 7 max epochs inside 1 round
    factor=3,
    directory="tuning",
    project_name="5g_mobilenet_bigru"
)

# ---- Tune ONLY on subset ----
sample_idx = np.random.choice(len(X_train), size=int(len(X_train)*0.25), replace=False)   # takes 25% of training data for hyperparameter tuning
X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]

stop_early = EarlyStopping(monitor='val_loss', patience=3)      # early stopping applied based on val_loss

tuner.search(
    X_tune, y_tune,
    validation_split=0.2,
    epochs=10,                                # total 10 rounds
    batch_size=512,
    callbacks=[stop_early],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]             # takes best hyperparameters

print("Best Hyperparameters:")
print(best_hps.values)


Trial 10 Complete [00h 04m 36s]
val_accuracy: 0.8654905557632446

Best val_accuracy So Far: 0.8736121654510498
Total elapsed time: 00h 23m 53s
Best Hyperparameters:
{'dropout': 0.3, 'gru_units': 128, 'dense': 256, 'lr': 0.001, 'tuner/epochs': 7, 'tuner/initial_epoch': 3, 'tuner/bracket': 1, 'tuner/round': 1, 'tuner/trial_id': '0001'}


**14. Build the Best Tuned Combined Model**

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


Epoch 1/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 91s 44ms/step - accuracy: 0.7066 - loss: 0.1117 - val_accuracy: 0.7704 - val_loss: 0.0933 - learning_rate: 0.0010
Epoch 2/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 45ms/step - accuracy: 0.8510 - loss: 0.0416 - val_accuracy: 0.8657 - val_loss: 0.0355 - learning_rate: 0.0010
Epoch 3/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8694 - loss: 0.0348 - val_accuracy: 0.8698 - val_loss: 0.0345 - learning_rate: 0.0010
Epoch 4/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8761 - loss: 0.0313 - val_accuracy: 0.8441 - val_loss: 0.0650 - learning_rate: 0.0010
Epoch 5/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8826 - loss: 0.0291 - val_accuracy: 0.8768 - val_loss: 0.0347 - learning_rate: 0.0010
Epoch 6/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accuracy: 0.8862 - loss: 0.0276 - val_accuracy: 0.8909 - val_loss: 0.0250 - learning_rate: 0.0010
Epoch 7/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 76s 44ms/step - accura

**15. Model Prediction on Test Dataset & Classification Report**

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))


7600/7600 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     18761
           1       0.97      0.97      0.97     18730
           2       0.68      0.92      0.78      3656
           3       0.84      0.78      0.81      2890
           4       0.33      0.65      0.43       322
           5       0.32      0.27      0.29       271
           6       0.98      0.94      0.96      8927
           7       0.94      0.93      0.94      2822
           8       0.94      0.89      0.92      2290
           9       0.94      0.88      0.91      2305
          10       0.82      0.81      0.82      6203
          11       0.57      0.58      0.58      2531
          12       0.58      0.08      0.14      2329
          13       0.51      0.88      0.64      2331
          14       0.97      0.94      0.95      7624
          15       0.98      0.96      0.97      6334
          16       0.99      0.86    

**16. Hyperparameter Tuning(Ranged) for Combined Model**

In [ ]:
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping

def build_model(hp):

    model = MobileNetV1_BiGRU(
        drop_rate = hp.Float(
            "dropout",
            min_value=0.2,             # hyperparameter tuning of dropout for range(0.2, 0.6)
            max_value=0.6,
            step=0.1
        ),

        gru_units = hp.Int(
            "gru_units",
            min_value=64,             # hyperparameter tuning of GRU units for range(64, 256)
            max_value=256,
            step=32
        ),

        dense_units = hp.Int(
            "dense",
            min_value=128,            # hyperparameter tuning of dense units for range(128, 512)
            max_value=512,
            step=64
        )
    )

    lr = hp.Float(
        "lr",
        min_value=1e-5,              # hyperparameter tuning of learning rate for range(1e-5, 1e-2)
        max_value=1e-2,
        sampling="log"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),          # adam optimizer
        loss=focal_loss(gamma=2, alpha=0.25),
        metrics=["accuracy"]
    )

    return model


tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=7,                      # max 7 epochs inside 1 round
    factor=3,
    directory="tuning",
    project_name="5g_mobilenet_bigru"
)

# ---- Tune ONLY on subset ----
sample_idx = np.random.choice(len(X_train), size=int(len(X_train)*1), replace=False)       # takes 100% of training data for hyperparameter tuning
X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]

stop_early = EarlyStopping(monitor='val_loss', patience=3)

tuner.search(
    X_tune,
    y_tune,
    validation_split=0.2,
    epochs=10,                      # 10 epochs
    batch_size=512,
    callbacks=[stop_early],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:")
print(best_hps.values)

Trial 10 Complete [00h 12m 56s]
val_accuracy: 0.8909305334091187

Best val_accuracy So Far: 0.896207869052887
Total elapsed time: 01h 01m 47s
Best Hyperparameters:
{'dropout': 0.2, 'gru_units': 96, 'dense': 256, 'lr': 0.0026970145809465584, 'tuner/epochs': 7, 'tuner/initial_epoch': 3, 'tuner/bracket': 1, 'tuner/round': 1, 'tuner/trial_id': '0000'}


**17. Build the Best Tuned Combined Model**

In [ ]:
model = tuner.hypermodel.build(best_hps)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=20,            # 20 epochs
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=8, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


Epoch 1/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 73s 39ms/step - accuracy: 0.7367 - loss: 0.0932 - val_accuracy: 0.8572 - val_loss: 0.0389 - learning_rate: 0.0027
Epoch 2/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 64s 37ms/step - accuracy: 0.8631 - loss: 0.0368 - val_accuracy: 0.8640 - val_loss: 0.0384 - learning_rate: 0.0027
Epoch 3/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.8752 - loss: 0.0316 - val_accuracy: 0.8901 - val_loss: 0.0257 - learning_rate: 0.0027
Epoch 4/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.8841 - loss: 0.0287 - val_accuracy: 0.8957 - val_loss: 0.0237 - learning_rate: 0.0027
Epoch 5/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 68s 40ms/step - accuracy: 0.8907 - loss: 0.0259 - val_accuracy: 0.8677 - val_loss: 0.0307 - learning_rate: 0.0027
Epoch 6/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 64s 38ms/step - accuracy: 0.8951 - loss: 0.0248 - val_accuracy: 0.9045 - val_loss: 0.0230 - learning_rate: 0.0027
Epoch 7/20
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accura

**18. Model Prediction on Test Dataset & Classification Report**

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))


7600/7600 ━━━━━━━━━━━━━━━━━━━━ 40s 5ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     18761
           1       0.98      0.99      0.98     18730
           2       0.86      0.92      0.89      3656
           3       0.83      0.91      0.87      2890
           4       0.48      0.58      0.52       322
           5       0.31      0.51      0.38       271
           6       0.97      0.95      0.96      8927
           7       0.95      0.96      0.95      2822
           8       0.98      0.90      0.94      2290
           9       0.96      0.88      0.92      2305
          10       0.91      0.97      0.94      6203
          11       0.85      0.82      0.84      2531
          12       0.51      0.35      0.42      2329
          13       0.49      0.65      0.56      2331
          14       0.99      0.95      0.97      7624
          15       0.98      0.98      0.98      6334
          16       0.97      0.98    

**19. New Hyperparameter Tuning(Ranged) for Combined Model**

In [ ]:
import keras_tuner as kt
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_model(hp):

    model = MobileNetV1_BiGRU(

        drop_rate = hp.Float(
            "dropout",
            min_value=0.2,
            max_value=0.6,
            step=0.1
        ),

        gru_units = hp.Int(
            "gru_units",
            min_value=64,
            max_value=256,
            step=32
        ),

        dense_units = hp.Int(
            "dense_units",
            min_value=128,
            max_value=512,
            step=64
        )
    )

    # focal loss parameters
    gamma = hp.Float(
        "gamma",
        min_value=1.0,
        max_value=5.0,
        step=0.5
    )

    alpha = hp.Float(
        "alpha",
        min_value=0.1,
        max_value=0.5,
        step=0.1
    )

    # learning rate
    lr = hp.Float(
        "lr",
        min_value=1e-5,
        max_value=1e-2,
        sampling="log"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )

    return model


# Hyperband tuner
tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=7,
    factor=3,
    directory="tuning",
    project_name="5g_mobilenet_bigru"
)


# Sample 75% data for tuning
sample_idx = np.random.choice(
    len(X_train),
    size=int(len(X_train) * 0.75),
    replace=False
)

X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]


stop_early = EarlyStopping(
    monitor="val_loss",
    patience=3
)


# Run hyperparameter search
tuner.search(
    X_tune,
    y_tune,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=[stop_early],
    verbose=1
)


# Get best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\nBest Hyperparameters Found:")

print("Dropout:", best_hps.get("dropout"))
print("GRU Units:", best_hps.get("gru_units"))
print("Dense Units:", best_hps.get("dense_units"))
print("Learning Rate:", best_hps.get("lr"))
print("Gamma:", best_hps.get("gamma"))
print("Alpha:", best_hps.get("alpha"))


# Tune epochs separately
best_epochs = best_hps.Int(
    "epochs",
    min_value=10,
    max_value=50,
    step=10
)

print("Epochs:", best_epochs)


Trial 10 Complete [00h 07m 06s]
val_accuracy: 0.8778125643730164

Best val_accuracy So Far: 0.8871335983276367
Total elapsed time: 01h 04m 09s

Best Hyperparameters Found:
Dropout: 0.4
GRU Units: 224
Dense Units: 384
Learning Rate: 0.002927382467972539
Gamma: 1.5
Alpha: 0.2
Epochs: 10


**20. Build the Best Tuned Combined Model**

In [ ]:
# Build best model
model = tuner.hypermodel.build(best_hps)


# Final training
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=best_epochs,
    batch_size=512,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)

Epoch 1/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 166s 93ms/step - accuracy: 0.7462 - loss: 0.0782 - val_accuracy: 0.8544 - val_loss: 0.0331 - learning_rate: 0.0029
Epoch 2/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 159s 93ms/step - accuracy: 0.8621 - loss: 0.0342 - val_accuracy: 0.8661 - val_loss: 0.0335 - learning_rate: 0.0029
Epoch 3/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 159s 93ms/step - accuracy: 0.8705 - loss: 0.0304 - val_accuracy: 0.8383 - val_loss: 0.0503 - learning_rate: 0.0029
Epoch 4/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 159s 93ms/step - accuracy: 0.8797 - loss: 0.0274 - val_accuracy: 0.8969 - val_loss: 0.0232 - learning_rate: 0.0029
Epoch 5/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 158s 92ms/step - accuracy: 0.8880 - loss: 0.0254 - val_accuracy: 0.8749 - val_loss: 0.0357 - learning_rate: 0.0029
Epoch 6/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 158s 92ms/step - accuracy: 0.8885 - loss: 0.0255 - val_accuracy: 0.8925 - val_loss: 0.0235 - learning_rate: 0.0029
Epoch 7/10
1710/1710 ━━━━━━━━━━━━━━━━━━━━ 157s 92ms/step -

**21. Model Prediction on Test Dataset & Classification Report**

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 42s 5ms/step
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     18761
           1       0.95      0.95      0.95     18730
           2       0.63      0.88      0.73      3656
           3       0.67      0.69      0.68      2890
           4       0.33      0.61      0.43       322
           5       0.36      0.18      0.25       271
           6       0.87      0.94      0.90      8927
           7       0.93      0.72      0.81      2822
           8       0.82      0.92      0.87      2290
           9       0.99      0.87      0.93      2305
          10       0.81      0.86      0.83      6203
          11       0.68      0.46      0.55      2531
          12       0.16      0.03      0.05      2329
          13       0.49      0.86      0.63      2331
          14       0.97      0.94      0.95      7624
          15       0.99      0.96      0.97      6334
          16       0.98      0.88    

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.2 MB/s eta 0:00:00


In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# sample 100% data for tuning
sample_idx = np.random.choice(
    len(X_train),
    size=int(len(X_train) * 0.5),
    replace=False
)

X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]


def objective(trial):

    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    # hyperparameters
    dropout = trial.suggest_float("dropout", 0.2, 0.5, step=0.1)

    gru_units = trial.suggest_int("gru_units", 64, 320, step=64)

    dense_units = trial.suggest_int("dense_units", 128, 448, step=64)

    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)

    alpha = trial.suggest_float("alpha", 0.2, 0.5, step=0.1)

    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)

    epochs = trial.suggest_int("epochs", 10, 20, step=10)


    # build model
    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )


    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]


    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )


    # objective value
    val_acc = max(history.history["val_accuracy"])

    return val_acc


# create study
study = optuna.create_study(direction="maximize")

optuna.logging.set_verbosity(optuna.logging.INFO)

# run optimization
study.optimize(objective, n_trials=10)


# best parameters
best_params = study.best_params

print("\nBest Hyperparameters Found:\n")

print("Dropout:", best_params["dropout"])
print("GRU Units:", best_params["gru_units"])
print("Dense Units:", best_params["dense_units"])
print("Learning Rate:", best_params["lr"])
print("Gamma:", best_params["gamma"])
print("Alpha:", best_params["alpha"])
print("Epochs:", best_params["epochs"])


# build final model with best parameters
model = MobileNetV1_BiGRU(
    drop_rate=best_params["dropout"],
    gru_units=best_params["gru_units"],
    dense_units=best_params["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
    loss=focal_loss(
        gamma=best_params["gamma"],
        alpha=best_params["alpha"]
    ),
    metrics=["accuracy"]
)



# final training on full dataset
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)

[I 2026-03-18 08:33:44,251] A new study created in memory with name: no-name-9cd2097b-6662-4335-95b0-36b80b768411



Starting Trial 0
Epoch 1/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 59s 26ms/step - accuracy: 0.7742 - loss: 0.0793 - val_accuracy: 0.8658 - val_loss: 0.0372 - learning_rate: 0.0020
Epoch 2/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 35s 23ms/step - accuracy: 0.8656 - loss: 0.0354 - val_accuracy: 0.8753 - val_loss: 0.0297 - learning_rate: 0.0020
Epoch 3/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 38s 25ms/step - accuracy: 0.8740 - loss: 0.0303 - val_accuracy: 0.8793 - val_loss: 0.0278 - learning_rate: 0.0020
Epoch 4/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 36s 24ms/step - accuracy: 0.8796 - loss: 0.0278 - val_accuracy: 0.8923 - val_loss: 0.0263 - learning_rate: 0.0020
Epoch 5/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 36s 23ms/step - accuracy: 0.8856 - loss: 0.0260 - val_accuracy: 0.8976 - val_loss: 0.0257 - learning_rate: 0.0020
Epoch 6/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 36s 24ms/step - accuracy: 0.8925 - loss: 0.0244 - val_accuracy: 0.8946 - val_loss: 0.0239 - learning_rate: 0.0020
Epoch 7/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 35s 

[I 2026-03-18 08:46:20,880] Trial 0 finished with value: 0.9442285299301147 and parameters: {'dropout': 0.2, 'gru_units': 64, 'dense_units': 512, 'gamma': 3.0, 'alpha': 0.4, 'lr': 0.001997908836813877, 'epochs': 20}. Best is trial 0 with value: 0.9442285299301147.



Starting Trial 1
Epoch 1/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 53s 31ms/step - accuracy: 0.5456 - loss: 0.2389 - val_accuracy: 0.6639 - val_loss: 0.1475 - learning_rate: 3.4400e-05
Epoch 2/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.6681 - loss: 0.1390 - val_accuracy: 0.7199 - val_loss: 0.1110 - learning_rate: 3.4400e-05
Epoch 3/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 45s 29ms/step - accuracy: 0.7107 - loss: 0.1104 - val_accuracy: 0.7360 - val_loss: 0.0929 - learning_rate: 3.4400e-05
Epoch 4/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.7330 - loss: 0.0948 - val_accuracy: 0.7621 - val_loss: 0.0826 - learning_rate: 3.4400e-05
Epoch 5/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.7512 - loss: 0.0856 - val_accuracy: 0.7899 - val_loss: 0.0751 - learning_rate: 3.4400e-05
Epoch 6/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 45s 30ms/step - accuracy: 0.7779 - loss: 0.0768 - val_accuracy: 0.8146 - val_loss: 0.0663 - learning_rate: 3.4400e-05
Epoch 7/20
1520/1520 ━

[I 2026-03-18 09:02:12,832] Trial 1 finished with value: 0.8990768194198608 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 384, 'gamma': 2.0, 'alpha': 0.30000000000000004, 'lr': 3.4399570120851916e-05, 'epochs': 20}. Best is trial 0 with value: 0.9442285299301147.


...


Starting Trial 8
Epoch 1/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 99s 61ms/step - accuracy: 0.7773 - loss: 0.1078 - val_accuracy: 0.8527 - val_loss: 0.0564 - learning_rate: 0.0010
Epoch 2/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 88s 58ms/step - accuracy: 0.8583 - loss: 0.0534 - val_accuracy: 0.8649 - val_loss: 0.0475 - learning_rate: 0.0010
Epoch 3/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 89s 59ms/step - accuracy: 0.8670 - loss: 0.0475 - val_accuracy: 0.8792 - val_loss: 0.0407 - learning_rate: 0.0010
Epoch 4/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 89s 59ms/step - accuracy: 0.8736 - loss: 0.0443 - val_accuracy: 0.8949 - val_loss: 0.0366 - learning_rate: 0.0010
Epoch 5/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 89s 58ms/step - accuracy: 0.8800 - loss: 0.0417 - val_accuracy: 0.9009 - val_loss: 0.0346 - learning_rate: 0.0010
Epoch 6/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 89s 58ms/step - accuracy: 0.8840 - loss: 0.0393 - val_accuracy: 0.9072 - val_loss: 0.0333 - learning_rate: 0.0010
Epoch 7/10
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 89s 

[I 2026-03-18 10:41:39,126] Trial 8 finished with value: 0.9073217511177063 and parameters: {'dropout': 0.5, 'gru_units': 256, 'dense_units': 512, 'gamma': 2.0, 'alpha': 0.4, 'lr': 0.0010391290832635935, 'epochs': 10}. Best is trial 6 with value: 0.946613609790802.



Starting Trial 9
Epoch 1/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 55s 31ms/step - accuracy: 0.6718 - loss: 0.1333 - val_accuracy: 0.7489 - val_loss: 0.0786 - learning_rate: 2.2075e-04
Epoch 2/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 48s 31ms/step - accuracy: 0.8017 - loss: 0.0653 - val_accuracy: 0.8432 - val_loss: 0.0481 - learning_rate: 2.2075e-04
Epoch 3/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 47s 31ms/step - accuracy: 0.8586 - loss: 0.0448 - val_accuracy: 0.8719 - val_loss: 0.0385 - learning_rate: 2.2075e-04
Epoch 4/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 47s 31ms/step - accuracy: 0.8715 - loss: 0.0385 - val_accuracy: 0.8907 - val_loss: 0.0341 - learning_rate: 2.2075e-04
Epoch 5/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 48s 31ms/step - accuracy: 0.8816 - loss: 0.0345 - val_accuracy: 0.8954 - val_loss: 0.0312 - learning_rate: 2.2075e-04
Epoch 6/20
1520/1520 ━━━━━━━━━━━━━━━━━━━━ 47s 31ms/step - accuracy: 0.8874 - loss: 0.0317 - val_accuracy: 0.9019 - val_loss: 0.0279 - learning_rate: 2.2075e-04
Epoch 7/20
1520/1520 ━

[I 2026-03-18 10:58:41,456] Trial 9 finished with value: 0.9392014145851135 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 448, 'gamma': 2.0, 'alpha': 0.30000000000000004, 'lr': 0.00022075062299324944, 'epochs': 20}. Best is trial 6 with value: 0.946613609790802.



Best Hyperparameters Found:

Dropout: 0.2
GRU Units: 128
Dense Units: 320
Learning Rate: 0.0013343885574274323
Gamma: 3.0
Alpha: 0.2
Epochs: 20
Epoch 1/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 109s 30ms/step - accuracy: 0.8250 - loss: 0.0287 - val_accuracy: 0.8771 - val_loss: 0.0152 - learning_rate: 0.0013
Epoch 2/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 101s 30ms/step - accuracy: 0.8781 - loss: 0.0152 - val_accuracy: 0.8774 - val_loss: 0.0132 - learning_rate: 0.0013
Epoch 3/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 102s 30ms/step - accuracy: 0.8890 - loss: 0.0133 - val_accuracy: 0.8932 - val_loss: 0.0116 - learning_rate: 0.0013
Epoch 4/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 101s 30ms/step - accuracy: 0.8954 - loss: 0.0120 - val_accuracy: 0.8934 - val_loss: 0.0100 - learning_rate: 0.0013
Epoch 5/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 142s 30ms/step - accuracy: 0.8988 - loss: 0.0114 - val_accuracy: 0.9053 - val_loss: 0.0104 - learning_rate: 0.0013
Epoch 6/20
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 102s 30ms/step - accuracy: 0

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     18761
           1       0.98      0.98      0.98     18730
           2       0.84      0.92      0.88      3656
           3       0.86      0.84      0.85      2890
           4       0.57      0.64      0.60       322
           5       0.44      0.67      0.53       271
           6       0.97      0.96      0.97      8927
           7       0.96      0.97      0.97      2822
           8       0.94      0.80      0.87      2290
           9       0.82      0.92      0.87      2305
          10       0.85      0.96      0.90      6203
          11       0.76      0.69      0.72      2531
          12       0.93      0.54      0.69      2329
          13       0.68      0.90      0.78      2331
          14       0.98      0.95      0.97      7624
          15       0.99      0.98      0.99      6334
          16       0.97      0.95    

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# sample 100% data for tuning
sample_idx = np.random.choice(
    len(X_train),
    size=int(len(X_train) * 1),
    replace=False
)

X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]


def objective(trial):

    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    # hyperparameters
    dropout = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)

    gru_units = trial.suggest_int("gru_units", 64, 192, step=64)

    dense_units = trial.suggest_int("dense_units", 256, 448, step=64)

    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)

    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)

    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)

    epochs = trial.suggest_int("epochs", 10, 30, step=10)


    # build model
    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )


    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]


    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )


    # objective value
    val_acc = max(history.history["val_accuracy"])

    return val_acc


# create study
study = optuna.create_study(direction="maximize")

optuna.logging.set_verbosity(optuna.logging.INFO)

# run optimization
study.optimize(objective, n_trials=10)


# best parameters
best_params = study.best_params

print("\nBest Hyperparameters Found:\n")

print("Dropout:", best_params["dropout"])
print("GRU Units:", best_params["gru_units"])
print("Dense Units:", best_params["dense_units"])
print("Learning Rate:", best_params["lr"])
print("Gamma:", best_params["gamma"])
print("Alpha:", best_params["alpha"])
print("Epochs:", best_params["epochs"])


# build final model with best parameters
model = MobileNetV1_BiGRU(
    drop_rate=best_params["dropout"],
    gru_units=best_params["gru_units"],
    dense_units=best_params["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
    loss=focal_loss(
        gamma=best_params["gamma"],
        alpha=best_params["alpha"]
    ),
    metrics=["accuracy"]
)



# final training on full dataset
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)

[I 2026-03-19 15:41:13,035] A new study created in memory with name: no-name-5127d4b7-5a8c-41f6-b27d-56364f5c38ed



Starting Trial 0
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 135s 36ms/step - accuracy: 0.7831 - loss: 0.0517 - val_accuracy: 0.8769 - val_loss: 0.0248 - learning_rate: 4.3924e-04
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8781 - loss: 0.0232 - val_accuracy: 0.8911 - val_loss: 0.0199 - learning_rate: 4.3924e-04
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 95s 31ms/step - accuracy: 0.8907 - loss: 0.0195 - val_accuracy: 0.8981 - val_loss: 0.0174 - learning_rate: 4.3924e-04
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 95s 31ms/step - accuracy: 0.9006 - loss: 0.0173 - val_accuracy: 0.8950 - val_loss: 0.0163 - learning_rate: 4.3924e-04
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 139s 30ms/step - accuracy: 0.9047 - loss: 0.0160 - val_accuracy: 0.9110 - val_loss: 0.0147 - learning_rate: 4.3924e-04
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 95s 31ms/step - accuracy: 0.9095 - loss: 0.0149 - val_accuracy: 0.9159 - val_loss: 0.0136 - learning_rate: 4.3924e-04
Epoch 7/10
3040/3040

[I 2026-03-19 15:58:23,212] Trial 0 finished with value: 0.9273528456687927 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 448, 'gamma': 2.0, 'alpha': 0.2, 'lr': 0.00043924239766766546, 'epochs': 10}. Best is trial 0 with value: 0.9273528456687927.



Starting Trial 1
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 98s 30ms/step - accuracy: 0.7998 - loss: 0.0976 - val_accuracy: 0.8671 - val_loss: 0.0534 - learning_rate: 4.3551e-04
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 91s 30ms/step - accuracy: 0.8849 - loss: 0.0477 - val_accuracy: 0.8981 - val_loss: 0.0417 - learning_rate: 4.3551e-04
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 91s 30ms/step - accuracy: 0.8974 - loss: 0.0411 - val_accuracy: 0.9111 - val_loss: 0.0366 - learning_rate: 4.3551e-04
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.9078 - loss: 0.0368 - val_accuracy: 0.9159 - val_loss: 0.0332 - learning_rate: 4.3551e-04
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.9125 - loss: 0.0342 - val_accuracy: 0.9145 - val_loss: 0.0328 - learning_rate: 4.3551e-04
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.9182 - loss: 0.0318 - val_accuracy: 0.9186 - val_loss: 0.0316 - learning_rate: 4.3551e-04
Epoch 7/10
3040/3040 ━

[I 2026-03-19 16:13:35,705] Trial 1 finished with value: 0.9387025237083435 and parameters: {'dropout': 0.2, 'gru_units': 128, 'dense_units': 320, 'gamma': 1.0, 'alpha': 0.30000000000000004, 'lr': 0.00043551366730255084, 'epochs': 10}. Best is trial 1 with value: 0.9387025237083435.


...


Starting Trial 4
Epoch 1/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 79s 24ms/step - accuracy: 0.7847 - loss: 0.0387 - val_accuracy: 0.8630 - val_loss: 0.0190 - learning_rate: 0.0011
Epoch 2/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - accuracy: 0.8636 - loss: 0.0181 - val_accuracy: 0.8736 - val_loss: 0.0154 - learning_rate: 0.0011
Epoch 3/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 71s 23ms/step - accuracy: 0.8750 - loss: 0.0151 - val_accuracy: 0.8899 - val_loss: 0.0126 - learning_rate: 0.0011
Epoch 4/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 72s 24ms/step - accuracy: 0.8818 - loss: 0.0137 - val_accuracy: 0.8908 - val_loss: 0.0118 - learning_rate: 0.0011
Epoch 5/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 73s 24ms/step - accuracy: 0.8900 - loss: 0.0126 - val_accuracy: 0.8999 - val_loss: 0.0111 - learning_rate: 0.0011
Epoch 6/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 72s 24ms/step - accuracy: 0.8945 - loss: 0.0119 - val_accuracy: 0.8870 - val_loss: 0.0113 - learning_rate: 0.0011
Epoch 7/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 71s 

[I 2026-03-19 17:47:11,959] Trial 4 finished with value: 0.9527508020401001 and parameters: {'dropout': 0.4, 'gru_units': 64, 'dense_units': 256, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.0011164718536017327, 'epochs': 30}. Best is trial 3 with value: 0.9586055278778076.



Starting Trial 5
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 140s 44ms/step - accuracy: 0.7647 - loss: 0.0589 - val_accuracy: 0.7871 - val_loss: 0.0484 - learning_rate: 0.0098
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 132s 43ms/step - accuracy: 0.7931 - loss: 0.0504 - val_accuracy: 0.7744 - val_loss: 0.0933 - learning_rate: 0.0098
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 131s 43ms/step - accuracy: 0.8013 - loss: 0.0485 - val_accuracy: 0.8107 - val_loss: 0.0429 - learning_rate: 0.0098
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 131s 43ms/step - accuracy: 0.8115 - loss: 0.0458 - val_accuracy: 0.8393 - val_loss: 0.0390 - learning_rate: 0.0098
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 43ms/step - accuracy: 0.8083 - loss: 0.0468 - val_accuracy: 0.7894 - val_loss: 0.0587 - learning_rate: 0.0098
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 142s 42ms/step - accuracy: 0.8103 - loss: 0.0470 - val_accuracy: 0.8169 - val_loss: 0.0436 - learning_rate: 0.0098
Epoch 7/10
3040/3040 ━━━━━━━━━━━━━━━━━━━

[I 2026-03-19 18:09:12,254] Trial 5 finished with value: 0.8756110668182373 and parameters: {'dropout': 0.4, 'gru_units': 192, 'dense_units': 448, 'gamma': 2.0, 'alpha': 0.2, 'lr': 0.009810676698720006, 'epochs': 10}. Best is trial 3 with value: 0.9586055278778076.



Starting Trial 6
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 80s 24ms/step - accuracy: 0.8272 - loss: 0.0549 - val_accuracy: 0.8696 - val_loss: 0.0331 - learning_rate: 0.0025
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 73s 24ms/step - accuracy: 0.8736 - loss: 0.0305 - val_accuracy: 0.8807 - val_loss: 0.0262 - learning_rate: 0.0025
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 74s 24ms/step - accuracy: 0.8859 - loss: 0.0269 - val_accuracy: 0.8993 - val_loss: 0.0234 - learning_rate: 0.0025
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 83s 25ms/step - accuracy: 0.8935 - loss: 0.0247 - val_accuracy: 0.9016 - val_loss: 0.0218 - learning_rate: 0.0025
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 72s 24ms/step - accuracy: 0.8970 - loss: 0.0232 - val_accuracy: 0.8994 - val_loss: 0.0223 - learning_rate: 0.0025
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 72s 24ms/step - accuracy: 0.9006 - loss: 0.0224 - val_accuracy: 0.9037 - val_loss: 0.0248 - learning_rate: 0.0025
Epoch 7/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 73s 

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# sample 100% data for tuning
sample_idx = np.random.choice(
    len(X_train),
    size=int(len(X_train) * 1),
    replace=False
)

X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]


def objective(trial):

    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    # hyperparameters
    dropout = trial.suggest_float("dropout", 0.2, 0.5, step=0.1)

    gru_units = trial.suggest_int("gru_units", 64, 192, step=64)

    dense_units = trial.suggest_int("dense_units", 256, 384, step=64)

    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)

    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    epochs = trial.suggest_int("epochs", 10, 30, step=10)


    # build model
    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )


    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]


    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )


    # objective value
    val_acc = max(history.history["val_accuracy"])

    return val_acc


# create study
study = optuna.create_study(direction="maximize")

optuna.logging.set_verbosity(optuna.logging.INFO)

# run optimization
study.optimize(objective, n_trials=10)


# best parameters
best_params = study.best_params

print("\nBest Hyperparameters Found:\n")

print("Dropout:", best_params["dropout"])
print("GRU Units:", best_params["gru_units"])
print("Dense Units:", best_params["dense_units"])
print("Learning Rate:", best_params["lr"])
print("Gamma:", best_params["gamma"])
print("Alpha:", best_params["alpha"])
print("Epochs:", best_params["epochs"])


# build final model with best parameters
model = MobileNetV1_BiGRU(
    drop_rate=best_params["dropout"],
    gru_units=best_params["gru_units"],
    dense_units=best_params["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
    loss=focal_loss(
        gamma=best_params["gamma"],
        alpha=best_params["alpha"]
    ),
    metrics=["accuracy"]
)



# final training on full dataset
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)

[I 2026-03-22 09:27:52,427] A new study created in memory with name: no-name-b64fe22e-3fbc-41b8-a292-d44d95358802



Starting Trial 0
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 145s 42ms/step - accuracy: 0.7735 - loss: 0.1503 - val_accuracy: 0.8776 - val_loss: 0.0774 - learning_rate: 2.5582e-04
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.8777 - loss: 0.0710 - val_accuracy: 0.8990 - val_loss: 0.0588 - learning_rate: 2.5582e-04
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.8927 - loss: 0.0594 - val_accuracy: 0.9077 - val_loss: 0.0543 - learning_rate: 2.5582e-04
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 134s 44ms/step - accuracy: 0.9005 - loss: 0.0536 - val_accuracy: 0.9102 - val_loss: 0.0485 - learning_rate: 2.5582e-04
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9089 - loss: 0.0488 - val_accuracy: 0.9175 - val_loss: 0.0436 - learning_rate: 2.5582e-04
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 133s 44ms/step - accuracy: 0.9150 - loss: 0.0454 - val_accuracy: 0.9262 - val_loss: 0.0395 - learning_rate: 2.5582e-04
Epoch 7/20
3040/

[I 2026-03-22 10:12:50,292] Trial 0 finished with value: 0.9838236570358276 and parameters: {'dropout': 0.2, 'gru_units': 192, 'dense_units': 320, 'gamma': 1.0, 'alpha': 0.4, 'lr': 0.00025581696553862954, 'epochs': 20}. Best is trial 0 with value: 0.9838236570358276.



Starting Trial 1
Epoch 1/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 100s 31ms/step - accuracy: 0.8347 - loss: 0.0733 - val_accuracy: 0.8798 - val_loss: 0.0471 - learning_rate: 0.0022
Epoch 2/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.8802 - loss: 0.0474 - val_accuracy: 0.8929 - val_loss: 0.0429 - learning_rate: 0.0022
Epoch 3/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 93s 31ms/step - accuracy: 0.8919 - loss: 0.0430 - val_accuracy: 0.9018 - val_loss: 0.0383 - learning_rate: 0.0022
Epoch 4/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 93s 31ms/step - accuracy: 0.8982 - loss: 0.0404 - val_accuracy: 0.9003 - val_loss: 0.0387 - learning_rate: 0.0022
Epoch 5/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 94s 31ms/step - accuracy: 0.9034 - loss: 0.0384 - val_accuracy: 0.9004 - val_loss: 0.0358 - learning_rate: 0.0022
Epoch 6/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 93s 31ms/step - accuracy: 0.9070 - loss: 0.0371 - val_accuracy: 0.9117 - val_loss: 0.0343 - learning_rate: 0.0022
Epoch 7/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 93s

[I 2026-03-22 11:00:14,964] Trial 1 finished with value: 0.9848054051399231 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 384, 'gamma': 1.0, 'alpha': 0.30000000000000004, 'lr': 0.0022052893576960772, 'epochs': 30}. Best is trial 1 with value: 0.9848054051399231.


...


Starting Trial 5
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 103s 32ms/step - accuracy: 0.7221 - loss: 0.0745 - val_accuracy: 0.8354 - val_loss: 0.0368 - learning_rate: 1.8922e-04
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 31ms/step - accuracy: 0.8573 - loss: 0.0308 - val_accuracy: 0.8801 - val_loss: 0.0243 - learning_rate: 1.8922e-04
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.8816 - loss: 0.0236 - val_accuracy: 0.8971 - val_loss: 0.0205 - learning_rate: 1.8922e-04
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.8933 - loss: 0.0205 - val_accuracy: 0.8960 - val_loss: 0.0185 - learning_rate: 1.8922e-04
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.9011 - loss: 0.0185 - val_accuracy: 0.9093 - val_loss: 0.0173 - learning_rate: 1.8922e-04
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.9064 - loss: 0.0171 - val_accuracy: 0.8989 - val_loss: 0.0163 - learning_rate: 1.8922e-04
Epoch 7/20
3040/3040 

[I 2026-03-22 13:46:05,928] Trial 5 finished with value: 0.9502886533737183 and parameters: {'dropout': 0.2, 'gru_units': 128, 'dense_units': 256, 'gamma': 2.0, 'alpha': 0.2, 'lr': 0.00018921512507970122, 'epochs': 20}. Best is trial 1 with value: 0.9848054051399231.



Starting Trial 6
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 83s 25ms/step - accuracy: 0.7847 - loss: 0.0780 - val_accuracy: 0.8655 - val_loss: 0.0400 - learning_rate: 6.8877e-04
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 81s 25ms/step - accuracy: 0.8708 - loss: 0.0344 - val_accuracy: 0.8921 - val_loss: 0.0287 - learning_rate: 6.8877e-04
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 76s 25ms/step - accuracy: 0.8821 - loss: 0.0293 - val_accuracy: 0.8835 - val_loss: 0.0262 - learning_rate: 6.8877e-04
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 75s 25ms/step - accuracy: 0.8898 - loss: 0.0259 - val_accuracy: 0.9004 - val_loss: 0.0229 - learning_rate: 6.8877e-04
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 82s 25ms/step - accuracy: 0.8959 - loss: 0.0236 - val_accuracy: 0.9061 - val_loss: 0.0216 - learning_rate: 6.8877e-04
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 76s 25ms/step - accuracy: 0.9021 - loss: 0.0217 - val_accuracy: 0.8994 - val_loss: 0.0215 - learning_rate: 6.8877e-04
Epoch 7/20
3040/3040 ━

[I 2026-03-22 14:11:59,593] Trial 6 finished with value: 0.9690402746200562 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 64, 'dense_units': 384, 'gamma': 3.0, 'alpha': 0.4, 'lr': 0.0006887681080237412, 'epochs': 20}. Best is trial 1 with value: 0.9848054051399231.



Starting Trial 7
Epoch 1/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 105s 32ms/step - accuracy: 0.8327 - loss: 0.0698 - val_accuracy: 0.8770 - val_loss: 0.0448 - learning_rate: 0.0030
Epoch 2/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 141s 32ms/step - accuracy: 0.8726 - loss: 0.0447 - val_accuracy: 0.8931 - val_loss: 0.0358 - learning_rate: 0.0030
Epoch 3/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 142s 32ms/step - accuracy: 0.8821 - loss: 0.0412 - val_accuracy: 0.8835 - val_loss: 0.0361 - learning_rate: 0.0030
Epoch 4/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 97s 32ms/step - accuracy: 0.8882 - loss: 0.0392 - val_accuracy: 0.8873 - val_loss: 0.0427 - learning_rate: 0.0030
Epoch 5/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 97s 32ms/step - accuracy: 0.9205 - loss: 0.0276 - val_accuracy: 0.9266 - val_loss: 0.0244 - learning_rate: 3.0219e-04
Epoch 6/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 96s 32ms/step - accuracy: 0.9312 - loss: 0.0241 - val_accuracy: 0.9417 - val_loss: 0.0220 - learning_rate: 3.0219e-04
Epoch 7/30
3040/3040 ━━━━━━━━━━━━━━

In [ ]:
model.save("final_model.h5")

In [ ]:
pred = model.predict(X_test)
pred = np.argmax(pred, axis=1)

print(classification_report(y_test, pred))

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     18761
           1       0.98      0.98      0.98     18730
           2       0.87      0.88      0.87      3656
           3       0.82      0.90      0.86      2890
           4       0.39      0.77      0.52       322
           5       0.35      0.45      0.40       271
           6       0.95      0.97      0.96      8927
           7       0.95      0.97      0.96      2822
           8       0.87      0.93      0.90      2290
           9       0.96      0.88      0.92      2305
          10       0.94      0.76      0.84      6203
          11       0.57      0.75      0.65      2531
          12       0.82      0.66      0.73      2329
          13       0.72      0.78      0.75      2331
          14       0.95      0.96      0.95      7624
          15       0.99      0.97      0.98      6334
          16       0.99      0.95    

**23. Optuna Training of hyperparameters**

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# sample 100% data for tuning
sample_idx = np.random.choice(
    len(X_train),
    size=int(len(X_train) * 1),
    replace=False
)

X_tune = X_train[sample_idx]
y_tune = y_train[sample_idx]


def objective(trial):

    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    # hyperparameters
    dropout = trial.suggest_float("dropout", 0.2, 0.5, step=0.1)

    gru_units = trial.suggest_int("gru_units", 64, 192, step=64)

    dense_units = trial.suggest_int("dense_units", 256, 384, step=64)

    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)

    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    epochs = trial.suggest_int("epochs", 10, 20, step=10)


    # build model
    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )


    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]


    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=epochs,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )


    # objective value
    val_acc = max(history.history["val_accuracy"])

    return val_acc


# create study
study = optuna.create_study(direction="maximize")

optuna.logging.set_verbosity(optuna.logging.INFO)

# run optimization
study.optimize(objective, n_trials=7)


# best parameters
best_params = study.best_params

print("\nBest Hyperparameters Found:\n")

print("Dropout:", best_params["dropout"])
print("GRU Units:", best_params["gru_units"])
print("Dense Units:", best_params["dense_units"])
print("Learning Rate:", best_params["lr"])
print("Gamma:", best_params["gamma"])
print("Alpha:", best_params["alpha"])
print("Epochs:", best_params["epochs"])


# build final model with best parameters
model = MobileNetV1_BiGRU(
    drop_rate=best_params["dropout"],
    gru_units=best_params["gru_units"],
    dense_units=best_params["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
    loss=focal_loss(
        gamma=best_params["gamma"],
        alpha=best_params["alpha"]
    ),
    metrics=["accuracy"]
)



# final training on full dataset
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=best_params["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(patience=4, restore_best_weights=True),
        ReduceLROnPlateau(patience=4)
    ]
)


model.save_weights("optuna_best_weights.h5")

[I 2026-03-31 08:37:00,206] A new study created in memory with name: no-name-e4a774ec-5899-47b0-9ea8-e667568f009c



Starting Trial 0
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 142s 41ms/step - accuracy: 0.7214 - loss: 0.1432 - val_accuracy: 0.8343 - val_loss: 0.0719 - learning_rate: 1.6819e-04
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 130s 43ms/step - accuracy: 0.8517 - loss: 0.0643 - val_accuracy: 0.8770 - val_loss: 0.0487 - learning_rate: 1.6819e-04
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 132s 43ms/step - accuracy: 0.8782 - loss: 0.0491 - val_accuracy: 0.8944 - val_loss: 0.0409 - learning_rate: 1.6819e-04
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 130s 43ms/step - accuracy: 0.8892 - loss: 0.0425 - val_accuracy: 0.8950 - val_loss: 0.0395 - learning_rate: 1.6819e-04
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 130s 43ms/step - accuracy: 0.8960 - loss: 0.0384 - val_accuracy: 0.8992 - val_loss: 0.0349 - learning_rate: 1.6819e-04
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 43ms/step - accuracy: 0.9013 - loss: 0.0355 - val_accuracy: 0.9104 - val_loss: 0.0317 - learning_rate: 1.6819e-04
Epoch 7/10
3040/

[I 2026-03-31 08:58:54,509] Trial 0 finished with value: 0.9244382977485657 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 192, 'dense_units': 384, 'gamma': 2.0, 'alpha': 0.4, 'lr': 0.00016819382626833478, 'epochs': 10}. Best is trial 0 with value: 0.9244382977485657.



Starting Trial 1
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 137s 43ms/step - accuracy: 0.8237 - loss: 0.0282 - val_accuracy: 0.8619 - val_loss: 0.0173 - learning_rate: 0.0017
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 42ms/step - accuracy: 0.8748 - loss: 0.0153 - val_accuracy: 0.8871 - val_loss: 0.0125 - learning_rate: 0.0017
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 43ms/step - accuracy: 0.8835 - loss: 0.0136 - val_accuracy: 0.8775 - val_loss: 0.0135 - learning_rate: 0.0017
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 128s 42ms/step - accuracy: 0.8901 - loss: 0.0125 - val_accuracy: 0.9022 - val_loss: 0.0104 - learning_rate: 0.0017
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 43ms/step - accuracy: 0.8968 - loss: 0.0118 - val_accuracy: 0.8959 - val_loss: 0.0106 - learning_rate: 0.0017
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 129s 42ms/step - accuracy: 0.9000 - loss: 0.0113 - val_accuracy: 0.9084 - val_loss: 0.0103 - learning_rate: 0.0017
Epoch 7/20
3040/3040 ━━━━━━━━━━━━━━━━━━━

[I 2026-03-31 09:41:50,959] Trial 1 finished with value: 0.9861572980880737 and parameters: {'dropout': 0.2, 'gru_units': 192, 'dense_units': 384, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.0016660841296666318, 'epochs': 20}. Best is trial 1 with value: 0.9861572980880737.


...


Starting Trial 4
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 98s 30ms/step - accuracy: 0.7983 - loss: 0.0735 - val_accuracy: 0.8785 - val_loss: 0.0329 - learning_rate: 7.0946e-04
Epoch 2/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.8793 - loss: 0.0320 - val_accuracy: 0.8859 - val_loss: 0.0280 - learning_rate: 7.0946e-04
Epoch 3/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.8916 - loss: 0.0268 - val_accuracy: 0.9048 - val_loss: 0.0224 - learning_rate: 7.0946e-04
Epoch 4/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.8983 - loss: 0.0239 - val_accuracy: 0.9061 - val_loss: 0.0219 - learning_rate: 7.0946e-04
Epoch 5/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 29ms/step - accuracy: 0.9055 - loss: 0.0217 - val_accuracy: 0.9100 - val_loss: 0.0206 - learning_rate: 7.0946e-04
Epoch 6/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.9076 - loss: 0.0205 - val_accuracy: 0.9125 - val_loss: 0.0190 - learning_rate: 7.0946e-04
Epoch 7/10
3040/3040 ━

[I 2026-03-31 10:52:37,979] Trial 4 finished with value: 0.9279748201370239 and parameters: {'dropout': 0.2, 'gru_units': 128, 'dense_units': 256, 'gamma': 3.0, 'alpha': 0.4, 'lr': 0.0007094573542823031, 'epochs': 10}. Best is trial 1 with value: 0.9861572980880737.



Starting Trial 5
Epoch 1/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 98s 30ms/step - accuracy: 0.7498 - loss: 0.1679 - val_accuracy: 0.8604 - val_loss: 0.0826 - learning_rate: 3.0849e-04
Epoch 2/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 91s 30ms/step - accuracy: 0.8657 - loss: 0.0787 - val_accuracy: 0.8898 - val_loss: 0.0621 - learning_rate: 3.0849e-04
Epoch 3/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 90s 30ms/step - accuracy: 0.8819 - loss: 0.0661 - val_accuracy: 0.9028 - val_loss: 0.0544 - learning_rate: 3.0849e-04
Epoch 4/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 93s 31ms/step - accuracy: 0.8913 - loss: 0.0597 - val_accuracy: 0.9086 - val_loss: 0.0508 - learning_rate: 3.0849e-04
Epoch 5/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 139s 30ms/step - accuracy: 0.8964 - loss: 0.0559 - val_accuracy: 0.9075 - val_loss: 0.0477 - learning_rate: 3.0849e-04
Epoch 6/20
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 91s 30ms/step - accuracy: 0.9017 - loss: 0.0526 - val_accuracy: 0.9127 - val_loss: 0.0445 - learning_rate: 3.0849e-04
Epoch 7/20
3040/3040 

[I 2026-03-31 11:25:30,172] Trial 5 finished with value: 0.9745249152183533 and parameters: {'dropout': 0.5, 'gru_units': 128, 'dense_units': 320, 'gamma': 1.0, 'alpha': 0.4, 'lr': 0.00030848804054207896, 'epochs': 20}. Best is trial 1 with value: 0.9861572980880737.



Starting Trial 6
Epoch 1/10
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 79s 24ms/step - accuracy: 0.8174 - loss: 0.0801 - val_accuracy: 0.8647 - val_loss: 0.0478 - learning_rate: 0.0051
Epoch 2/10
2239/3040 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.8533 - loss: 0.0577

**24. Compiled model**

 Trial 1 finished with value: 0.9861572980880737 and parameters: {'dropout': 0.2, 'gru_units': 192, 'dense_units': 384, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.0016660841296666318, 'epochs': 20}

In [ ]:
def get_compiled_model():

    model = MobileNetV1_BiGRU(
        drop_rate=best_params["dropout"],
        gru_units=best_params["gru_units"],
        dense_units=best_params["dense_units"]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
        loss=focal_loss(
            gamma=best_params["gamma"],
            alpha=best_params["alpha"]
        ),
        metrics=["accuracy"]
    )

    return model

**25. Create Federated Clients**

In [ ]:
NUM_CLIENTS = 3

client_data = {}

X_split = np.array_split(X_train, NUM_CLIENTS)          # (training) Data is distributed almost equally between the 5 clients
y_split = np.array_split(y_train, NUM_CLIENTS)

for i in range(NUM_CLIENTS):
    client_data[i] = (X_split[i], y_split[i])

**26. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=2):

    # compiled model
    local_model = get_compiled_model()

    # load global weights
    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=256,
        verbose=0
    )

    return local_model.get_weights()

**27. Federated Averaging (FedAvg)**

In [ ]:
def aggregate_weights(client_weights):

    avg_weights = []

    for i, weights_list_tuple in enumerate(zip(*client_weights)):
        layer_avg = np.mean(weights_list_tuple, axis=0)

        print(f"\nLayer {i} aggregation:")
        print(f"Shape: {layer_avg.shape}")
        print(f"Mean: {np.mean(layer_avg):.6f}")
        print(f"Std: {np.std(layer_avg):.6f}")

        avg_weights.append(layer_avg)

    return avg_weights

**28. Federated Training Loop (Server)**

In [ ]:
import random

ROUNDS = 5
CLIENTS_PER_ROUND = 3

# global model (compiled)
global_model = get_compiled_model()

# load pretrained weights
global_model.load_weights("optuna_best_weights.h5")

weights = global_model.get_weights()

print("First layer mean:", np.mean(weights[0]))

for round_num in range(ROUNDS):

    print("\nFL Round:", round_num)

    global_weights = global_model.get_weights()

    client_weights = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:

        weights = client_update(
            global_weights,
            client_data[client]
        )

        client_weights.append(weights)

    new_global_weights = aggregate_weights(client_weights)

    global_model.set_weights(new_global_weights)

    # ADD THIS
    loss, acc = global_model.evaluate(X_test, y_test, verbose=1)

    print(f"Global Accuracy after round {round_num}: {acc:.4f}")

**29. Evaluate Global Model**

In [ ]:
loss, accuracy = global_model.evaluate(X_test, y_test)
print("Final Global Model Accuracy:", accuracy)

In [ ]:
# 🔹 manually set best hyperparameters (replace with your values)
BEST_PARAMS = {
    "dropout": 0.30000000000000004,
    "gru_units": 128,
    "dense_units": 384,
    "lr": 0.0022052893576960772,
    "gamma": 1.0,
    "alpha": 0.30000000000000004,
    "epochs": 30
}

tf.keras.backend.clear_session()

model = MobileNetV1_BiGRU(
    drop_rate=BEST_PARAMS["dropout"],
    gru_units=BEST_PARAMS["gru_units"],
    dense_units=BEST_PARAMS["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=BEST_PARAMS["lr"]
    ),
    loss=focal_loss(
        gamma=BEST_PARAMS["gamma"],
        alpha=BEST_PARAMS["alpha"]
    ),
    metrics=["accuracy"]
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=BEST_PARAMS["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=4,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            patience=4
        )
    ],
    verbose=1
)

Epoch 1/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 110s 29ms/step - accuracy: 0.8377 - loss: 0.0719 - val_accuracy: 0.8773 - val_loss: 0.0462 - learning_rate: 0.0022
Epoch 2/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 101s 29ms/step - accuracy: 0.8809 - loss: 0.0476 - val_accuracy: 0.8915 - val_loss: 0.0402 - learning_rate: 0.0022
Epoch 3/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 99s 29ms/step - accuracy: 0.8903 - loss: 0.0426 - val_accuracy: 0.8870 - val_loss: 0.0385 - learning_rate: 0.0022
Epoch 4/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 99s 29ms/step - accuracy: 0.8976 - loss: 0.0402 - val_accuracy: 0.9138 - val_loss: 0.0345 - learning_rate: 0.0022
Epoch 5/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accuracy: 0.9043 - loss: 0.0378 - val_accuracy: 0.9088 - val_loss: 0.0346 - learning_rate: 0.0022
Epoch 6/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accuracy: 0.9071 - loss: 0.0370 - val_accuracy: 0.8971 - val_loss: 0.0403 - learning_rate: 0.0022
Epoch 7/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accu

In [ ]:
# 🔹 manually set best hyperparameters (replace with your values)
BEST_PARAMS = {
    "dropout": 0.2,
    "gru_units": 192,
    "dense_units": 384,
    "lr": 0.0016660841296666318,
    "gamma": 3.0,
    "alpha": 0.2,
    "epochs": 30
}

tf.keras.backend.clear_session()

model = MobileNetV1_BiGRU(
    drop_rate=BEST_PARAMS["dropout"],
    gru_units=BEST_PARAMS["gru_units"],
    dense_units=BEST_PARAMS["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=BEST_PARAMS["lr"]
    ),
    loss=focal_loss(
        gamma=BEST_PARAMS["gamma"],
        alpha=BEST_PARAMS["alpha"]
    ),
    metrics=["accuracy"]
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=BEST_PARAMS["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=4,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            patience=4
        )
    ],
    verbose=1
)

Epoch 1/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 148s 40ms/step - accuracy: 0.8296 - loss: 0.0270 - val_accuracy: 0.8924 - val_loss: 0.0140 - learning_rate: 0.0017
Epoch 2/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 140s 41ms/step - accuracy: 0.8784 - loss: 0.0152 - val_accuracy: 0.8872 - val_loss: 0.0138 - learning_rate: 0.0017
Epoch 3/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 142s 41ms/step - accuracy: 0.8868 - loss: 0.0135 - val_accuracy: 0.8976 - val_loss: 0.0114 - learning_rate: 0.0017
Epoch 4/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 147s 43ms/step - accuracy: 0.8935 - loss: 0.0125 - val_accuracy: 0.9029 - val_loss: 0.0104 - learning_rate: 0.0017
Epoch 5/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 143s 42ms/step - accuracy: 0.8985 - loss: 0.0119 - val_accuracy: 0.9092 - val_loss: 0.0107 - learning_rate: 0.0017
Epoch 6/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 140s 41ms/step - accuracy: 0.9008 - loss: 0.0115 - val_accuracy: 0.8888 - val_loss: 0.0100 - learning_rate: 0.0017
Epoch 7/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 142s 41ms/step -

In [ ]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f"Validation Accuracy: {val_acc:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.9311 - loss: 0.0082
Validation Accuracy: 0.9311


In [ ]:
# 🔹 manually set best hyperparameters (replace with your values)
BEST_PARAMS = {
    "dropout": 0.30000000000000004,
    "gru_units": 128,
    "dense_units": 320,
    "lr": 0.0006915204635387746,
    "gamma": 3.0,
    "alpha": 0.4,
    "epochs": 30
}

tf.keras.backend.clear_session()

model = MobileNetV1_BiGRU(
    drop_rate=BEST_PARAMS["dropout"],
    gru_units=BEST_PARAMS["gru_units"],
    dense_units=BEST_PARAMS["dense_units"]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=BEST_PARAMS["lr"]
    ),
    loss=focal_loss(
        gamma=BEST_PARAMS["gamma"],
        alpha=BEST_PARAMS["alpha"]
    ),
    metrics=["accuracy"]
)


history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=BEST_PARAMS["epochs"],
    batch_size=256,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=4,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            patience=4
        )
    ],
    verbose=1
)

Epoch 1/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 114s 29ms/step - accuracy: 0.8007 - loss: 0.0722 - val_accuracy: 0.8773 - val_loss: 0.0325 - learning_rate: 6.9152e-04
Epoch 2/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 99s 29ms/step - accuracy: 0.8754 - loss: 0.0330 - val_accuracy: 0.8900 - val_loss: 0.0306 - learning_rate: 6.9152e-04
Epoch 3/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accuracy: 0.8857 - loss: 0.0279 - val_accuracy: 0.8980 - val_loss: 0.0228 - learning_rate: 6.9152e-04
Epoch 4/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accuracy: 0.8936 - loss: 0.0249 - val_accuracy: 0.9024 - val_loss: 0.0230 - learning_rate: 6.9152e-04
Epoch 5/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 99s 29ms/step - accuracy: 0.8999 - loss: 0.0227 - val_accuracy: 0.9136 - val_loss: 0.0189 - learning_rate: 6.9152e-04
Epoch 6/30
3420/3420 ━━━━━━━━━━━━━━━━━━━━ 98s 29ms/step - accuracy: 0.9044 - loss: 0.0212 - val_accuracy: 0.9155 - val_loss: 0.0170 - learning_rate: 6.9152e-04
Epoch 7/30
3420/3420 ━━━━━━━━━━━━━━━━━━

In [ ]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f"Validation Accuracy: {val_acc:.4f}")

7600/7600 ━━━━━━━━━━━━━━━━━━━━ 52s 7ms/step - accuracy: 0.9369 - loss: 0.0132
Validation Accuracy: 0.9369
